# Run NLF on the 10demo dataset and export SMPL-X parameters

Notebook version of `run_nlf_smplx.py`.

For a given person (e.g. `100831`), every camera folder under
`<data-root>/<person>/images/cam_XX` holds 60 frames (`0025.jpg` ... `1500.jpg`,
step 25) of one continuous motion filmed from a fixed camera.

This notebook feeds each camera's frames to the NLF (Neural Localizer Fields)
model **in chronological order** and does simple nearest-box tracking across
frames so the same person is followed throughout the sequence, rather than
picking an unrelated detection in a frame that happens to contain more than
one candidate box.

**How to use:** just run all cells top to bottom (Kernel -> Restart & Run All).
All paths default relative to this notebook's folder (assumed to be the repo
root, next to `10demo/`), and the NLF checkpoint is downloaded automatically
into `models/` on first run. Edit the *Configuration* cell below if you want
to change the person, cameras, batch size, etc.

Requires `torch`, `torchvision` and `numpy` in the kernel's environment.

## 1. Imports

- `torch` / `torchvision` run the NLF TorchScript model. Note that `torchvision`
  must be imported even though we call it indirectly -- the traced model
  references torchvision ops internally and fails to load without it.
- `numpy` is used for all the per-frame array bookkeeping (boxes, poses, ...).
- `pathlib.Path` is used throughout instead of raw strings for all filesystem
  paths, since it works the same way on Windows and Linux.
- `urllib.request` downloads the model checkpoint if it isn't there yet.
- `from __future__ import annotations` lets us write modern type hints like
  `str | None` while keeping them valid across Python versions (it just
  changes how annotations are evaluated, not the runtime behavior).

In [ ]:
from __future__ import annotations

import re
import sys
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torchvision  # noqa: F401  (required for the traced model to load correctly)
from torchvision.io import ImageReadMode, read_image

## 2. Configuration

These are the same knobs that `run_nlf_smplx.py` exposes as `--flags` on the
command line. In the notebook they're just plain variables you can edit
directly and re-run.

- `REPO_ROOT` is resolved from the current working directory. Jupyter starts
  a notebook's working directory at the folder the `.ipynb` file lives in, so
  as long as this notebook stays at the repo root (next to `10demo/`), this
  is correct. If you move the notebook, change `REPO_ROOT` accordingly.
- `DATA_ROOT` / `PERSON` locate the input images: `10demo/main/100831/images/cam_XX`.
- `OUTPUT_ROOT` is kept **independent of the `10demo/` folder** (a sibling
  directory), so the generated `.npz` files don't get mixed in with the raw
  dataset. Results land in `OUTPUT_ROOT/<person>/cam_XX/<frame>.npz`.
- `MODEL_PATH` / `MODEL_URL` control the NLF checkpoint. If `MODEL_PATH`
  doesn't exist, it's downloaded from `MODEL_URL` automatically (a real
  release asset from https://github.com/isarandi/nlf, about 500 MB).
- `CAMERAS`: leave as `None` to process all 16 `cam_00..cam_15` folders found,
  or set to a list like `['cam_00', 'cam_05']` to restrict to a subset.
- `BATCH_SIZE` is how many frames are pushed through the network in one
  forward pass; lower it if you hit GPU out-of-memory errors.
- `DETECTOR_THRESHOLD` / `NUM_AUG` / `BETA_REGULARIZER` are passed straight
  through to NLF's `detect_smpl_batched` (detection confidence cutoff, number
  of test-time augmentation crops per person, and the SMPL-X shape-parameter
  regularization strength).
- `OVERWRITE`: set to `True` to force recomputing frames whose output `.npz`
  already exists (by default, already-saved frames are skipped so the
  notebook can be safely re-run / resumed).

In [ ]:
REPO_ROOT = Path.cwd()

DATA_ROOT = REPO_ROOT / '10demo' / 'main'
PERSON = '100831'
OUTPUT_ROOT = REPO_ROOT / 'nlf_smplx_params'  # independent of the 10demo/ folder

MODEL_PATH = REPO_ROOT / 'models' / 'nlf_l_multi_0.3.2.torchscript'
MODEL_URL = 'https://github.com/isarandi/nlf/releases/download/v0.3.2/nlf_l_multi_0.3.2.torchscript'

CAMERAS = None  # e.g. ['cam_00', 'cam_01'] to restrict to a subset; None = all found
DEVICE = None   # e.g. 'cuda' or 'cpu'; None = auto-pick cuda if available

BATCH_SIZE = 16
DETECTOR_THRESHOLD = 0.3
NUM_AUG = 5
BETA_REGULARIZER = 10.0
OVERWRITE = False

# Frame filenames look like '0025.jpg'. '1500.jpg'; this excludes the
# trailing 'A-2045.jpg' file, which is not part of the continuous sequence.
FRAME_RE = re.compile(r'^(\d{4})\.jpg$')

## 3. SMPL-X pose layout

NLF's `detect_smpl_batched(..., model_name='smplx')` returns, per detected
person, a single flat `pose` vector of length `55 * 3 = 165`: one axis-angle
rotation (3 numbers) for each of the 55 SMPL-X joints, concatenated in a
fixed order. `split_smplx_pose` cuts that vector back into the named parts
(`global_orient`, `body_pose`, `jaw_pose`, `leye_pose`, `reye_pose`,
`left_hand_pose`, `right_hand_pose`) so the saved `.npz` files are easy to
feed into an `smplx.SMPLX(...)` body model later, e.g.:

```python
res = body_model(global_orient=..., body_pose=..., left_hand_pose=...)
```

Note these hand poses are full 45-dim axis-angle (NLF's native output), not
the 6-component PCA hand poses used by the *original* ground-truth
`smplx_params/*.npz` files already shipped with the dataset -- the two are
not directly interchangeable without a PCA basis conversion.

In [ ]:
N_GLOBAL = 3       # global_orient: 1 joint
N_BODY = 21 * 3    # body_pose: 21 joints
N_JAW = 3          # jaw_pose: 1 joint
N_LEYE = 3         # leye_pose: 1 joint
N_REYE = 3         # reye_pose: 1 joint
N_LHAND = 15 * 3   # left_hand_pose: 15 joints
N_RHAND = 15 * 3   # right_hand_pose: 15 joints


def split_smplx_pose(pose: np.ndarray) -> dict:
    """Slice a flat 165-dim SMPL-X axis-angle pose vector into named parts."""
    i = 0
    parts = {}
    for name, n in [
        ('global_orient', N_GLOBAL),
        ('body_pose', N_BODY),
        ('jaw_pose', N_JAW),
        ('leye_pose', N_LEYE),
        ('reye_pose', N_REYE),
        ('left_hand_pose', N_LHAND),
        ('right_hand_pose', N_RHAND),
    ]:
        parts[name] = pose[i:i + n]
        i += n
    assert i == pose.shape[0], f'expected 165-dim SMPL-X pose, got {pose.shape[0]}'
    return parts

## 4. Model checkpoint download

`ensure_model` checks whether the TorchScript checkpoint already exists at
`MODEL_PATH`. If not, it streams it down from `MODEL_URL` into a temporary
`.part` file (so a crash or interrupted download never leaves a corrupt file
at the final path) with a simple textual progress readout, then renames it
into place.

In [ ]:
def ensure_model(model_path: Path, model_url: str) -> Path:
    """Download the NLF checkpoint into place if it isn't there yet."""
    if model_path.exists():
        return model_path

    model_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'Model checkpoint not found at {model_path}.')
    print(f'Downloading it from {model_url} (this is a one-time ~500MB download) ...')

    tmp_path = model_path.with_suffix(model_path.suffix + '.part')

    def report(block_num, block_size, total_size):
        if total_size <= 0:
            return
        done = min(block_num * block_size, total_size)
        pct = 100 * done / total_size
        sys.stdout.write(f'\r  {done / 1e6:8.1f} / {total_size / 1e6:.1f} MB ({pct:5.1f}%)')
        sys.stdout.flush()

    try:
        urllib.request.urlretrieve(model_url, tmp_path, reporthook=report)
        print()
    except Exception:
        tmp_path.unlink(missing_ok=True)
        raise
    tmp_path.rename(model_path)
    print(f'Saved model to {model_path}')
    return model_path

## 5. Locating cameras and frames

- `list_camera_dirs` looks inside `<person>/images/` for subfolders named
  `cam_XX` and returns them sorted (`cam_00`, `cam_01`, ...); pass a list of
  names via `CAMERAS` above to restrict to a subset.
- `list_frame_paths` lists the frames of a single camera folder, keeping only
  files that match the strict `NNNN.jpg` pattern (this is what excludes the
  extra `A-2045.jpg` file) and sorts them **numerically** by frame index --
  this ordering is what makes the sequence "continuous" for the tracking
  step later on.

In [ ]:
def list_camera_dirs(person_images_dir: Path, cameras: list[str] | None) -> list[Path]:
    all_cams = sorted(
        p for p in person_images_dir.iterdir() if p.is_dir() and p.name.startswith('cam_')
    )
    if not cameras:
        return all_cams
    wanted = set(cameras)
    selected = [p for p in all_cams if p.name in wanted]
    missing = wanted - {p.name for p in selected}
    if missing:
        raise FileNotFoundError(f'Requested camera(s) not found: {sorted(missing)}')
    return selected


def list_frame_paths(camera_dir: Path) -> list[Path]:
    frames = []
    for p in camera_dir.iterdir():
        m = FRAME_RE.match(p.name)
        if m:
            frames.append((int(m.group(1)), p))
    frames.sort(key=lambda t: t[0])
    return [p for _, p in frames]

## 6. Tracking the same person across frames

NLF's detector can return more than one bounding box per frame (e.g. a false
positive, or a bystander walking through the shot). Since we know each
camera sequence follows *one* continuous motion, we track a single identity
through the frames with a simple nearest-box heuristic instead of trusting
detection order:

- **First frame with a detection:** pick the box with the highest detector
  confidence score (5th column of the box array).
- **Every subsequent frame:** pick whichever detected box's center is
  closest to the *previous* frame's chosen box center. Because a person
  moves smoothly and continuously between nearby frames, this keeps the
  same identity locked on even if the detector also fires on something else.

Boxes are in `[x, y, w, h, score]` format (top-left corner, width, height,
confidence).

In [ ]:
def box_center(box: np.ndarray) -> np.ndarray:
    x, y, w, h = box[:4]
    return np.array([x + w / 2, y + h / 2])


def pick_tracked_index(boxes: np.ndarray, prev_box: np.ndarray | None) -> int:
    """Pick which detected person to keep, favoring continuity with the previous frame."""
    if prev_box is None:
        return int(np.argmax(boxes[:, 4]))  # highest detector score
    prev_c = box_center(prev_box)
    centers = np.stack([box_center(b) for b in boxes])
    dists = np.linalg.norm(centers - prev_c, axis=1)
    return int(np.argmin(dists))

## 7. Loading a batch of frames

`load_image_batch` reads a list of image files with `torchvision.io.read_image`
(decoding straight to a `uint8` CxHxW tensor, no manual normalization needed --
NLF's TorchScript model expects raw uint8 pixel values and normalizes
internally), stacks them into a single `[B, 3, H, W]` batch, and moves that
batch to the target device (GPU or CPU). All frames from the same camera
share resolution, so stacking them is safe; this is checked explicitly.

In [ ]:
def load_image_batch(paths: list[Path], device: torch.device) -> torch.Tensor:
    images = [read_image(str(p), mode=ImageReadMode.RGB) for p in paths]
    shapes = {tuple(im.shape) for im in images}
    if len(shapes) != 1:
        raise RuntimeError(
            f'Frames in the same camera must share resolution, got shapes {shapes} for {paths}'
        )
    return torch.stack(images).to(device)

## 8. Processing one camera's sequence

`process_camera` is the core loop, run once per camera:

1. Frames are split into chunks of `batch_size` (to bound GPU memory), but
   always processed **in chronological order**, so tracking state
   (`prev_box`) carries over correctly from one chunk to the next.
2. If every frame in a chunk already has a saved `.npz` (from a previous run)
   and `overwrite=False`, the chunk is skipped -- but `prev_box` is still
   refreshed from the last saved file, so tracking continuity is preserved
   across a resumed run.
3. Otherwise, the chunk is loaded and pushed through
   `model.detect_smpl_batched(..., model_name='smplx')`, which returns, per
   frame, a list of candidate `boxes`, `pose`, `betas`, and `trans` tensors
   (one entry per person NLF detected in that frame).
4. For each frame: if nothing was detected, the frame is recorded as
   `missing` and skipped. Otherwise `pick_tracked_index` selects which
   detection to keep, and its SMPL-X `pose` (165-dim), `betas` (10-dim, shape
   coefficients) and `trans` (3-dim, root translation) are written to
   `<out_dir>/<frame>.npz`, together with the tracked `box` and the split-out
   named pose fields from `split_smplx_pose`.

In [ ]:
def process_camera(
    model,
    frame_paths: list[Path],
    out_dir: Path,
    device: torch.device,
    batch_size: int,
    detector_threshold: float,
    num_aug: int,
    beta_regularizer: float,
    overwrite: bool,
) -> tuple[int, list[str]]:
    out_dir.mkdir(parents=True, exist_ok=True)
    prev_box = None
    n_saved = 0
    missing = []

    for chunk_start in range(0, len(frame_paths), batch_size):
        chunk_paths = frame_paths[chunk_start:chunk_start + batch_size]

        # Skip frames whose output already exists, unless overwriting -- but we still
        # need to refresh `prev_box` so tracking continuity is preserved if a later
        # chunk in the same camera does need to run.
        if not overwrite and all(
            (out_dir / f'{p.stem}.npz').exists() for p in chunk_paths
        ):
            for p in chunk_paths:
                data = np.load(out_dir / f'{p.stem}.npz')
                prev_box = data['box']
            n_saved += len(chunk_paths)
            continue

        frame_batch = load_image_batch(chunk_paths, device)
        with torch.inference_mode():
            pred = model.detect_smpl_batched(
                frame_batch,
                model_name='smplx',
                detector_threshold=detector_threshold,
                num_aug=num_aug,
                beta_regularizer=beta_regularizer,
            )

        for path, boxes_t, pose_t, betas_t, trans_t in zip(
            chunk_paths, pred['boxes'], pred['pose'], pred['betas'], pred['trans']
        ):
            boxes = boxes_t.detach().cpu().numpy()
            if boxes.shape[0] == 0:
                missing.append(path.name)
                continue

            idx = pick_tracked_index(boxes, prev_box)
            prev_box = boxes[idx]

            pose = pose_t[idx].detach().cpu().numpy().astype(np.float32)
            betas = betas_t[idx].detach().cpu().numpy().astype(np.float32)
            trans = trans_t[idx].detach().cpu().numpy().astype(np.float32)
            parts = split_smplx_pose(pose)

            np.savez(
                out_dir / f'{path.stem}.npz',
                pose=pose,
                betas=betas,
                trans=trans,
                box=prev_box.astype(np.float32),
                frame=path.stem,
                **parts,
            )
            n_saved += 1

    return n_saved, missing

## 9. Setup: resolve paths, pick device, load the model

This cell:

1. Builds `images_dir` from `DATA_ROOT`/`PERSON` and checks it exists.
2. Picks the compute device: whatever `DEVICE` was set to above, or CUDA if
   available, else CPU.
3. Calls `ensure_model` (downloading the checkpoint on first run).
4. Loads the TorchScript model with `torch.jit.load` and moves it to the
   chosen device in `eval()` mode (no dropout/batchnorm training behavior,
   and no gradients needed for inference).

In [ ]:
person_dir = DATA_ROOT / PERSON
images_dir = person_dir / 'images'
if not images_dir.is_dir():
    raise FileNotFoundError(f'Could not find images dir: {images_dir}')

output_dir = OUTPUT_ROOT / PERSON
device = torch.device(DEVICE) if DEVICE else torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

model_path = ensure_model(MODEL_PATH, MODEL_URL)

print(f'Loading model from {model_path} onto {device} ...')
model = torch.jit.load(str(model_path), map_location=device).to(device).eval()
print('Model loaded.')

## 10. Run: process every camera for this person

Loops over every camera folder found (or the subset given in `CAMERAS`),
lists its frames in order, and calls `process_camera` on each -- printing a
short progress line per camera and a final summary of how many `.npz` files
were written and which frames (if any) had no detection.

In [ ]:
camera_dirs = list_camera_dirs(images_dir, CAMERAS)
if not camera_dirs:
    raise FileNotFoundError(f'No camera folders found under {images_dir}')

print(f'Found {len(camera_dirs)} camera(s) for person {PERSON}: '
      f'{[c.name for c in camera_dirs]}')

total_saved = 0
all_missing = {}
for camera_dir in camera_dirs:
    frame_paths = list_frame_paths(camera_dir)
    if not frame_paths:
        print(f'  [{camera_dir.name}] no frames found, skipping')
        continue
    print(f'  [{camera_dir.name}] processing {len(frame_paths)} frames '
          f'({frame_paths[0].name} .. {frame_paths[-1].name}) ...')
    n_saved, missing = process_camera(
        model,
        frame_paths,
        output_dir / camera_dir.name,
        device,
        BATCH_SIZE,
        DETECTOR_THRESHOLD,
        NUM_AUG,
        BETA_REGULARIZER,
        OVERWRITE,
    )
    total_saved += n_saved
    if missing:
        all_missing[camera_dir.name] = missing
        print(f'  [{camera_dir.name}] WARNING: no person detected in {len(missing)} '
              f'frame(s): {missing}')

print(f'Done. Saved {total_saved} SMPL-X parameter files under {output_dir}')
if all_missing:
    print('Frames with no detection (left unsaved):')
    for cam, frames in all_missing.items():
        print(f'  {cam}: {frames}')

## 11. (Optional) Sanity-check a saved result

Quick smoke test that loads back one of the `.npz` files just written and
prints the shape of every array, so you can confirm the output looks as
expected before using it downstream.

In [ ]:
sample_cam_dir = output_dir / camera_dirs[0].name
sample_files = sorted(sample_cam_dir.glob('*.npz'))
if sample_files:
    sample = np.load(sample_files[0])
    print(f'Sample file: {sample_files[0]}')
    for key in sample.files:
        print(f'  {key}: shape={sample[key].shape}, dtype={sample[key].dtype}')
else:
    print(f'No saved files found in {sample_cam_dir}')